# 📗 Notebook 2 — Qwen3-8B
## Extraction de définitions juridiques maritimes (FR + EN)

### Pourquoi Qwen3-8B ?
- **Champion de l'extraction structurée** : Qwen3 domine les benchmarks d'Information Extraction (IE) sur HuggingFace Open LLM Leaderboard 2024/2025, notamment IFEval et BBH
- **JSON natif** : entraîné avec un focus sur la génération JSON fiable, minimisant les erreurs de parsing
- **Multilingue de haut niveau** : supporte 29 langues dont FR, EN, AR — excellent pour les textes juridiques mixtes
- **Fenêtre contextuelle 128K** : la plus grande parmi les 3 modèles, idéale pour les conventions longues (ICRW, CBD)
- **Licence Apache 2.0** : usage libre
- **Comparison point** : son architecture Transformer améliorée (GQA, SwiGLU) le rend plus précis que Mistral sur les listes numérotées complexes


In [1]:
# ─── Installation des dépendances ───────────────────────────────────────────
!pip install -q transformers accelerate bitsandbytes pdfplumber PyMuPDF pandas tqdm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 2.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.9/67.9 kB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 30.9 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 110.8 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.9/24.9 MB 74.6 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 92.0 MB/s eta 0:00:00:00:01


In [16]:
import os
os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"  # FIX OOM: réduit la fragmentation VRAM

import torch
import json
import re
import time
import pandas as pd
from pathlib import Path
from tqdm.auto import tqdm
import pdfplumber
import fitz
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

print(f"GPU disponible : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU : {torch.cuda.get_device_name(0)}")
    print(f"VRAM : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

GPU disponible : True
GPU : Tesla T4
VRAM : 15.6 GB


In [3]:
# ─── Configuration ───────────────────────────────────────────────────────────
MODEL_NAME = "Qwen/Qwen3-8B"
OUTPUT_DIR = Path("/kaggle/working/output/qwen")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

DOCUMENTS = [
    {"id": "I001", "label": "Chalutage de fond", "lang": "fr",
     "path": "/kaggle/input/datasets/assoumanasouley/chalutage-fond/clalut_fond.pdf"},
    {"id": "I002", "label": "Chasse à la baleine", "lang": "en",
     "path": "/kaggle/input/datasets/assoumanasouley/convention/raw/raw/Baleine/ICRW_convention.pdf"},
    {"id": "I003", "label": "Construction littorale", "lang": "fr",
     "path": "/kaggle/input/datasets/assoumanasouley/convention/raw/raw/construction_sur_le_littorale/protocole_GIZC.pdf"},
    {"id": "I004", "label": "Extraction de sable", "lang": "fr",
     "path": "/kaggle/input/datasets/assoumanasouley/convention/raw/raw/extraction_Sable/convention_cbd.pdf"},
    {"id": "I005a", "label": "Rejet hydrocarbures - Protocole", "lang": "fr",
     "path": "/kaggle/input/datasets/assoumanasouley/convention/raw/raw/Rejet_dhydrocarbure/94ig4_4_protocol_fre.pdf"},
    {"id": "I005b", "label": "Rejet hydrocarbures - MARPOL Annexe I", "lang": "fr",
     "path": "/kaggle/input/datasets/assoumanasouley/convention/raw/raw/Rejet_dhydrocarbure/Annexe1consolidee_Marpol.pdf"},
    {"id": "I006", "label": "Sacs plastique - MARPOL Annexe V", "lang": "fr",
     "path": "/kaggle/input/datasets/assoumanasouley/convention/raw/raw/Sac_plastique/ANNEXEV_marpol.pdf"},
    {"id": "I007", "label": "Peintures TBT - AFS Convention", "lang": "en",
     "path": "/kaggle/input/datasets/assoumanasouley/convention/raw/raw/TBT/AFS_convention.pdf"},
    {"id": "I008", "label": "Oiseaux marins - CMS", "lang": "fr",
     "path": "/kaggle/input/datasets/assoumanasouley/convention/raw/raw/Oiseaux_marin/CMS_oiseau_marin.pdf"},
]

In [4]:
# ─── Extraction PDF ──────────────────────────────────────────────────────────
def extract_pdf_text(pdf_path: str, max_chars: int = 28000) -> str:
    """
    Qwen3 supporte 128K tokens → on peut extraire plus de texte.
    Stratégie : extraction complète + nettoyage des sauts de page.
    """
    text = ""
    try:
        with pdfplumber.open(pdf_path) as pdf:
            for i, page in enumerate(pdf.pages):
                page_text = page.extract_text()
                if page_text:
                    text += f"\n--- Page {i+1} ---\n{page_text}"
        if len(text.strip()) < 100:
            raise ValueError("PDF scanné détecté")
    except Exception:
        try:
            doc = fitz.open(pdf_path)
            for page in doc:
                text += page.get_text() + "\n"
        except Exception as e:
            print(f"  [ERROR] {e}")
            return ""
    
    text = re.sub(r'\n{4,}', '\n\n\n', text)
    text = re.sub(r'\f', '\n\n', text)  # Supprime les form feeds
    return text[:max_chars]


# Test rapide
test_doc = DOCUMENTS[0]
if Path(test_doc["path"]).exists():
    sample = extract_pdf_text(test_doc["path"])
    print(f"Extrait ({len(sample)} chars)")
else:
    print("[INFO] Fichier non trouvé (normal en dev local)")

Extrait (28000 chars)


In [8]:
# ─── Prompts — Qwen utilise le format ChatML ──────────────────────────────────
# Qwen3 répond mieux avec un system prompt explicite + schema JSON dans le user message

SYSTEM_PROMPT_FR = (
    "Tu es un expert en droit maritime international spécialisé dans l'extraction d'information. "
    "Tu extrais uniquement les données demandées au format JSON strict, sans commentaire ni explication. "
    "Tu ne paraphrases JAMAIS — tu copies le texte original exact."
)

SYSTEM_PROMPT_EN = (
    "You are an international maritime law expert specialized in information extraction. "
    "You extract only the requested data in strict JSON format, without comments or explanations. "
    "You NEVER paraphrase — you copy the exact original text."
)

USER_PROMPT_FR = """Extrait toutes les définitions de ce document juridique.

Les définitions apparaissent dans des sections : « Définitions », « Interprétation », « Règle 1 »,
ou introduites par : signifie, désigne, s'entend de, est défini comme, Aux fins de la présente,
ou dans des listes numérotées/lettrées sous un article définitionnel.
Les termes peuvent être entre guillemets « », en gras, en italique.

Schéma JSON attendu (tableau d'objets) :
[
  {{
    "terme": "<terme exact>",
    "definition": "<définition littérale complète>",
    "nom_scientifique": "<nom entre parenthèses ou null>",
    "reference": "<Article X / Règle Y / §Z>",
    "exclusions": "<exclusions/restrictions ou null>",
    "langue": "fr"
  }}
]

Réponds UNIQUEMENT avec le tableau JSON. Aucun texte avant ou après.

DOCUMENT :
{text}"""

USER_PROMPT_EN = """Extract all definitions from this legal document.

Definitions appear in sections: "Definitions", "Interpretation", "Rule 1",
or introduced by: means, shall mean, refers to, is defined as, for the purposes of,
or in numbered/lettered lists under a definitional article.
Terms may appear in quotes, bold, or italic.

Expected JSON schema (array of objects):
[
  {{
    "terme": "<exact term>",
    "definition": "<complete literal definition>",
    "nom_scientifique": "<name in parentheses or null>",
    "reference": "<Article X / Rule Y / §Z>",
    "exclusions": "<exclusions/restrictions or null>",
    "langue": "en"
  }}
]

Respond ONLY with the JSON array. No text before or after.

DOCUMENT:
{text}"""

def get_messages(lang: str, text: str) -> list:
    if lang == "fr":
        return [
            {"role": "system", "content": SYSTEM_PROMPT_FR},
            {"role": "user", "content": USER_PROMPT_FR.format(text=text)}
        ]
    else:
        return [
            {"role": "system", "content": SYSTEM_PROMPT_EN},
            {"role": "user", "content": USER_PROMPT_EN.format(text=text)}
        ]

In [9]:
# ─── Chargement Qwen3-8B en 4-bit ─────────────────────────────────────────
print(f"Chargement de {MODEL_NAME}...")

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,  # bfloat16 recommandé pour Qwen3
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.bfloat16,
    trust_remote_code=True,
)
model.eval()
print("✅ Qwen3-8B chargé")
if torch.cuda.is_available():
    print(f"VRAM utilisée : {torch.cuda.memory_allocated()/1e9:.2f} GB")

Chargement de Qwen/Qwen3-8B...


config.json:   0%|          | 0.00/728 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/399 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

✅ Qwen3-8B chargé
VRAM utilisée : 1.54 GB


In [18]:
# ─── Parser JSON robuste (partagé) ──────────────────────────────────────────
def parse_json_response(response: str, doc_id: str, model_label: str = "qwen3-8b") -> list:
    """Parser JSON robuste avec 3 stratégies de fallback."""
    clean = re.sub(r'```(?:json)?\s*', '', response)
    clean = re.sub(r'```\s*', '', clean).strip()

    # Stratégie 1 : parse direct
    try:
        data = json.loads(clean)
        if isinstance(data, list):
            for item in data:
                item['doc_id'] = doc_id
                item['modele'] = model_label
            return data
    except json.JSONDecodeError:
        pass

    # Stratégie 2 : extraire le tableau JSON (greedy)
    match = re.search(r'(\[.*\])', clean, re.DOTALL)
    if match:
        try:
            data = json.loads(match.group(1))
            if isinstance(data, list):
                for item in data:
                    item['doc_id'] = doc_id
                    item['modele'] = model_label
                return data
        except json.JSONDecodeError:
            pass

    # Stratégie 3 : JSON tronqué → récupération partielle
    truncated = re.search(r'(\[.*)', clean, re.DOTALL)
    if truncated:
        fragment = truncated.group(1).rstrip(',').rstrip() + ']' 
        try:
            data = json.loads(fragment)
            if isinstance(data, list) and data:
                print(f"  [WARNING] JSON tronqué pour {doc_id}, récupération partielle : {len(data)} objet(s)")
                for item in data:
                    item['doc_id'] = doc_id
                    item['modele'] = model_label
                return data
        except json.JSONDecodeError:
            pass

    # Stratégie 4 : extraction par objets individuels
    objects = re.findall(r'\{[^{}]+\}', clean, re.DOTALL)
    valid_items = []
    for obj_str in objects:
        try:
            item = json.loads(obj_str)
            if 'terme' in item or 'term' in item:
                item['doc_id'] = doc_id
                item['modele'] = model_label
                valid_items.append(item)
        except json.JSONDecodeError:
            pass

    if valid_items:
        print(f"  [FALLBACK] {len(valid_items)} objets extraits par regex pour {doc_id}")
        return valid_items

    print(f"  [WARNING] Aucun JSON trouvé pour {doc_id}")
    print(f"  Réponse brute : {response[:300]}")
    return []


def extract_relevant_sections(text: str) -> str:
    """
    Extrait les sections contenant des définitions juridiques.
    Stratégie 1 : sections titrées (Définitions, Interprétation...)
    Stratégie 2 : patterns inline (terme signifie / means)
    Stratégie 3 : listes numérotées/lettrées
    Fallback : retourne le texte complet si rien détecté.
    """
    collected = []

    # ── Stratégie 1 : sections titrées (toutes casses) ───────────────────────
    title_patterns = [
        r'((?:Article|ARTICLE|Rule|RULE|Règle 1|RÈGLE|Section|SECTION|Paragraph|PARAGRAPH|Para\.?)\s*[\dIVXivx]+\s*[-–.]?\s*(?:DÉFINITIONS?|Définitions?|définitions?|DEFINITIONS?|Definitions?|definitions?|INTERPRÉTATION|Interprétation|interprétation|INTERPRETATION|Interpretation|interpretation|MEANING|Meaning|meaning)\b.*?)(?=\n(?:Article|ARTICLE|Rule|RULE|Règle|RÈGLE|Section|SECTION|Paragraph|PARAGRAPH)\s*[\dIVXivx]+\b)',
        r'((?:DÉFINITIONS?|Définitions?|définitions?|DEFINITIONS?|Definitions?|definitions?|INTERPRÉTATION|Interprétation|interprétation|INTERPRETATION|Interpretation|interpretation|MEANING|Meaning|meaning)\s*\n+.*?)(?=\n{2,}[A-ZÀÂÉÈÊËÎÏÔÙÛÜ]{2}|\nArticle|\nARTICLE|\nRule|\nRULE|\nSection|\nSECTION)',
    ]
    for pattern in title_patterns:
        matches = re.findall(pattern, text, re.DOTALL)
        collected.extend(matches)

    # ── Stratégie 2 : pattern "terme" means/signifie ─────────────────────────
    trigger = (
        r'(?:'
        r'[Mm][Ee][Aa][Nn][Ss]?\b'
        r'|[Ss][Hh][Aa][Ll][Ll]\s+[Mm][Ee][Aa][Nn]\b'
        r'|[Rr][Ee][Ff][Ee][Rr][Ss]?\s+[Tt][Oo]\b'
        r'|[Ii][Ss]\s+[Dd][Ee][Ff][Ii][Nn][Ee][Dd]\s+[Aa][Ss]\b'
        r'|[Ss][Ii][Gg][Nn][Ii][Ff][Ii][Ee]\b'
        r'|[Dd][Éé][Ss][Ii][Gg][Nn][Ee]\b'
        r'|[Ss][\""][Ee][Nn][Tt][Ee][Nn][Dd]\s+(?:[Dd][Ee]|[Cc][Oo][Mm][Mm][Ee])\b'
        r'|[Ee][Ss][Tt]\s+[Dd][Éé][Ff][Ii][Nn][Ii]\s+[Cc][Oo][Mm][Mm][Ee]\b'
        r'|[Aa][Uu][Xx]\s+[Ff][Ii][Nn][Ss]\s+(?:[Dd][Ee]\s+[Ll][Aa]\s+[Pp][Rr][Éé][Ss][Ee][Nn][Tt][Ee]|[Dd][Uu]\s+[Pp][Rr][Éé][Ss][Ee][Nn][Tt])\b'
        r')' 
    )
    term = (
        r'(?:'
        r'«\s*[^»]{1,80}\s*»'
        r'|\"\s*[^\"]{1,80}\s*\"'
        r'|\u201c\s*[^\u201d]{1,80}\s*\u201d'
        r'|\u201e[^\u201d]{1,80}\u201d'
        r"|'[^']{1,80}'"
        r'|[A-ZÀÂÉÈÊËÎÏÔÙÛÜ][A-ZÀÂÉÈÊËÎÏÔÙÛÜ\s\-]{2,50}(?=\s)'
        r'|[A-ZÀÂÉÈÊËÎÏÔÙÛÜ][a-zàâéèêëîïôùûüa-z\s\-]{2,50}(?=\s)'
        r')' 
    )
    pattern_inline = re.compile(
        rf'({term}\s*(?:\([^)]*\))?\s*{trigger}.*?)(?=\n\s*\n|\Z|{term}\s*(?:\([^)]*\))?\s*{trigger})',
        re.DOTALL
    )
    collected.extend(pattern_inline.findall(text))

    # ── Stratégie 3 : listes numérotées/lettrées ─────────────────────────────
    pattern_list = re.compile(
        rf'(?:^\s*(?:\d+\.|[a-zA-Z]\)|[ivxIVX]+\.)\s+)({term}\s*(?:\([^)]*\))?\s*{trigger}.*?)(?=\n\s*(?:\d+\.|[a-zA-Z]\)|[ivxIVX]+\.)\s|\n\s*\n|\Z)',
        re.DOTALL | re.MULTILINE
    )
    collected.extend(pattern_list.findall(text))

    if collected:
        seen, deduped = set(), []
        for item in collected:
            key = item.strip()[:100]
            if key not in seen:
                seen.add(key)
                deduped.append(item.strip())
        result = "\n\n".join(deduped)
        if len(result) < 200:
            print(f"  [WARNING] Sections trop courtes ({len(result)} chars), chunking document entier")
            return text
        print(f"  [INFO] {len(deduped)} bloc(s) trouvé(s) → {len(result)} chars ciblés")
        return result

    print(f"  [WARNING] Aucune section détectée, chunking document entier")
    return text

# ─── Inférence Qwen3 sur un chunk ────────────────────────────────────────────
def _infer_qwen_chunk(text_chunk: str, lang: str, doc_id: str) -> list:
    """Appelle Qwen3 sur un chunk de texte et retourne les définitions parsées."""
    # FIX: libérer la VRAM entre chaque chunk pour éviter l'OOM
    import gc
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.synchronize()
    messages = get_messages(lang, text_chunk)
    text_input = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=False,  # FIX: désactive le mode "thinking" de Qwen3
    )
    model_inputs = tokenizer([text_input], return_tensors="pt").to(model.device)
    input_len = model_inputs["input_ids"].shape[1]

    with torch.inference_mode():
        generated_ids = model.generate(
            **model_inputs,
            max_new_tokens=2048,  # FIX: réduit pour économiser la VRAM (2048 suffisant pour JSON)
            temperature=0.05,
            do_sample=True,
            top_p=0.95,
            repetition_penalty=1.05,
            pad_token_id=tokenizer.eos_token_id,  # FIX: évite le warning "attention_mask"
        )
    new_ids = generated_ids[0][input_len:]
    response = tokenizer.decode(new_ids, skip_special_tokens=True).strip()
    return parse_json_response(response, doc_id, "qwen3-8b")


# ─── Extraction principale avec chunking ─────────────────────────────────────
def extract_definitions_qwen(doc_text: str, lang: str, doc_id: str) -> list:
    CHUNK_SIZE = 2800   # T4 15GB + Qwen3-8B 4-bit : max safe ~2800 chars (~700 tokens + prompt)

    targeted = extract_relevant_sections(doc_text)

    if len(targeted) <= CHUNK_SIZE:
        print(f"  [INFO] Sections ciblées ({len(targeted)} chars)")
        chunks = [targeted]
    else:
        print(f"  [INFO] Chunking ({len(targeted)} chars)")
        paragraphs = targeted.split("\n\n")
        chunks, current = [], ""
        for para in paragraphs:
            # FIX: si un paragraphe dépasse CHUNK_SIZE, le découper par tranches de caractères
            if len(para) > CHUNK_SIZE:
                if current:
                    chunks.append(current.strip())
                    current = ""
                for i in range(0, len(para), CHUNK_SIZE):
                    chunks.append(para[i:i + CHUNK_SIZE].strip())
                continue
            if len(current) + len(para) + 2 > CHUNK_SIZE and current:
                chunks.append(current.strip())
                current = para
            else:
                current += ("\n\n" if current else "") + para
        if current:
            chunks.append(current.strip())

    print(f"  [INFO] {len(chunks)} chunk(s) à traiter")

    all_defs = []
    for idx, chunk in enumerate(chunks, 1):
        print(f"    → Chunk {idx}/{len(chunks)} ({len(chunk)} chars)...", end=" ", flush=True)
        start = time.time()
        defs = _infer_qwen_chunk(chunk, lang, doc_id)
        elapsed = time.time() - start
        print(f"{len(defs)} définitions ({elapsed:.1f}s)")
        all_defs.extend(defs)

    seen, unique_defs = set(), []
    for d in all_defs:
        key = (str(d.get("terme", "")).strip().lower(), str(d.get("definition", ""))[:80])
        if key not in seen:
            seen.add(key)
            unique_defs.append(d)

    print(f"  [INFO] Total après dédoublonnage : {len(unique_defs)} définitions")
    return unique_defs

    
        


In [19]:
# ─── Pipeline d'extraction avec chunking ────────────────────────────────────
all_definitions = []
results_by_doc = {}
timing_stats = []

for doc in tqdm(DOCUMENTS, desc="[Qwen3] Extraction"):
    doc_path = Path(doc["path"])
    print(f"\n{'='*60}")
    print(f"[{doc['id']}] {doc['label']} ({doc['lang'].upper()})")

    if not doc_path.exists():
        print(f"  [SKIP] Fichier introuvable")
        continue

    text = extract_pdf_text(str(doc_path))
    if not text:
        print(f"  [SKIP] Texte vide (PDF scanné sans OCR ?)")
        continue
    print(f"  Texte extrait : {len(text)} caractères")

    t0 = time.time()
    defs = extract_definitions_qwen(text, doc["lang"], doc["id"])
    total_time = time.time() - t0

    print(f"  → {len(defs)} définitions extraites en {total_time:.1f}s")
    timing_stats.append({"doc_id": doc["id"], "time_s": round(total_time, 1), "nb_defs": len(defs)})

    all_definitions.extend(defs)
    results_by_doc[doc["id"]] = {
        "label": doc["label"], "lang": doc["lang"],
        "nb_definitions": len(defs), "definitions": defs
    }

    with open(OUTPUT_DIR / f"{doc['id']}_definitions.json", "w", encoding="utf-8") as f:
        json.dump(defs, f, ensure_ascii=False, indent=2)

print(f"\n✅ Total : {len(all_definitions)} définitions extraites sur {len(DOCUMENTS)} documents")


[Qwen3] Extraction:   0%|          | 0/9 [00:00<?, ?it/s]


[I001] Chalutage de fond (FR)
  Texte extrait : 28000 caractères
  [WARNING] Aucune section détectée, chunking document entier
  [INFO] Chunking (28000 chars)
  [INFO] 10 chunk(s) à traiter
    → Chunk 1/10 (2799 chars)... 23 définitions (114.3s)
    → Chunk 2/10 (2800 chars)... 8 définitions (52.3s)
    → Chunk 3/10 (2799 chars)... 20 définitions (153.8s)
    → Chunk 4/10 (2800 chars)... 26 définitions (133.9s)
    → Chunk 5/10 (2800 chars)... 0 définitions (11.8s)
    → Chunk 6/10 (2800 chars)... 0 définitions (11.9s)
    → Chunk 7/10 (2800 chars)... 0 définitions (9.0s)
    → Chunk 8/10 (2800 chars)... 14 définitions (74.8s)
    → Chunk 9/10 (2800 chars)... 12 définitions (63.1s)
    → Chunk 10/10 (2800 chars)...   [FALLBACK] 51 objets extraits par regex pour I001
51 définitions (281.7s)
  [INFO] Total après dédoublonnage : 147 définitions
  → 147 définitions extraites en 906.7s

[I002] Chasse à la baleine (EN)
  Texte extrait : 28000 caractères
  [INFO] 24 bloc(s) trouvé(s) → 2671

In [20]:
# ─── Sauvegarde & statistiques ───────────────────────────────────────────────
with open(OUTPUT_DIR / "all_definitions_qwen.json", "w", encoding="utf-8") as f:
    json.dump(all_definitions, f, ensure_ascii=False, indent=2)

if all_definitions:
    df = pd.DataFrame(all_definitions)
    df.to_csv(OUTPUT_DIR / "all_definitions_qwen.csv", index=False, encoding="utf-8-sig")

    print("📊 Résumé Qwen3-8B :")
    display(df.head(10))

    print("\n⏱️ Statistiques de temps :")
    df_timing = pd.DataFrame(timing_stats)
    display(df_timing)
    print(f"Temps total : {df_timing['time_s'].sum():.1f}s")

    print("\n📈 Métriques d'extraction (Qwen3-8B) :")
    for col in ['terme', 'definition', 'reference', 'nom_scientifique', 'exclusions']:
        if col in df.columns:
            null_pct = df[col].isnull().mean() * 100
            print(f"  - {col} : {null_pct:.1f}% manquants")
    if 'definition' in df.columns:
        print(f"  - Longueur moy. définitions : {df['definition'].str.len().mean():.0f} chars")
    if 'langue' in df.columns:
        print(f"  - Distribution par langue :\n{df['langue'].value_counts().to_string()}")
    if 'doc_id' in df.columns:
        print(f"  - Distribution par document :\n{df['doc_id'].value_counts().to_string()}")

    print("\n📊 Résumé par document :")
    summary = [{"id": k, "label": v["label"], "lang": v["lang"], "nb_definitions": v["nb_definitions"]}
               for k, v in results_by_doc.items()]
    display(pd.DataFrame(summary))
else:
    print("Aucune définition extraite.")


📊 Résumé Qwen3-8B :


,terme,definition,reference,exclusions,langue,doc_id,modele,nom_scientifique,exchanges
0,CSITEP,Classification statistique internationale type...,Page 1,None,fr,I001,qwen3-8b,NaN,NaN
1,CWP,Groupe de travail chargé de coordonner les sta...,Page 1,None,fr,I001,qwen3-8b,NaN,NaN
2,FAO,Organisation des Nations Unies pour l'alimenta...,Page 3,None,fr,I001,qwen3-8b,NaN,NaN
3,INDNR,"Pêche illicite, non déclarée et non réglementée",Page 1,None,fr,I001,qwen3-8b,NaN,NaN
4,engins de pêche,Les engins de pêche typiques,Page 1,None,fr,I001,qwen3-8b,NaN,NaN
5,captures de pêche,Les captures de pêche effectuées par les diffé...,Page 1,None,fr,I001,qwen3-8b,NaN,NaN
6,permis et autorisations des opérations de pêche,Les permis et les autorisations des opérations...,Page 1,None,fr,I001,qwen3-8b,NaN,NaN
7,statistiques de pêche,Les statistiques de pêche,Page 1,None,fr,I001,qwen3-8b,NaN,NaN
8,pêches commerciales,Les pêches commerciales,Page 1,None,fr,I001,qwen3-8b,NaN,NaN
9,pêches de subsistance,Les pêches de subsistance,Page 1,None,fr,I001,qwen3-8b,NaN,NaN



⏱️ Statistiques de temps :


,doc_id,time_s,nb_defs
0,I001,906.7,147
1,I002,767.3,90
2,I003,55.2,6
3,I004,594.7,67
4,I005a,2.6,0
5,I005b,956.0,53
6,I006,1335.4,126
7,I007,653.8,65
8,I008,891.5,115


Temps total : 6163.2s

📈 Métriques d'extraction (Qwen3-8B) :
  - terme : 0.0% manquants
  - definition : 0.0% manquants
  - reference : 7.2% manquants
  - nom_scientifique : 99.7% manquants
  - exclusions : 98.7% manquants
  - Longueur moy. définitions : 139 chars
  - Distribution par langue :
langue
fr    487
en    155
  - Distribution par document :
doc_id
I001     147
I006     126
I008     115
I002      90
I004      67
I007      65
I005b     53
I003       6

📊 Résumé par document :


,id,label,lang,nb_definitions
0,I001,Chalutage de fond,fr,147
1,I002,Chasse à la baleine,en,90
2,I003,Construction littorale,fr,6
3,I004,Extraction de sable,fr,67
4,I005a,Rejet hydrocarbures - Protocole,fr,0
5,I005b,Rejet hydrocarbures - MARPOL Annexe I,fr,53
6,I006,Sacs plastique - MARPOL Annexe V,fr,126
7,I007,Peintures TBT - AFS Convention,en,65
8,I008,Oiseaux marins - CMS,fr,115
